<a href="https://colab.research.google.com/github/Vinod1403/IIIT-Hyderabad-Tutorials/blob/main/Copy_of_STP_AIML_Module6_Lab4_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent Neural Networks with PyTorch



In [1]:
import torch
from torch import nn
import numpy as np

In [2]:
text = ['hey how are you','good i am fine','have a nice day']
chars = set(''.join(text))
int2char = dict(enumerate(chars))
char2int = {char: ind for ind, char in int2char.items()}

In [3]:
print(char2int)

{'y': 0, 'd': 1, 'f': 2, 'w': 3, 'e': 4, 'o': 5, 'i': 6, 'c': 7, ' ': 8, 'a': 9, 'n': 10, 'h': 11, 'r': 12, 'g': 13, 'u': 14, 'v': 15, 'm': 16}


In [4]:
maxlen = len(max(text, key=len))
print("The longest string has {} characters".format(maxlen))

The longest string has 15 characters


In [5]:
for i in range(len(text)):
    while len(text[i])<maxlen:
        text[i] += ' '

In [6]:
input_seq = []
target_seq = []
for i in range(len(text)):
    input_seq.append(text[i][:-1])
    target_seq.append(text[i][1:])
    print("Input Sequence: {}\nTarget Sequence: {}".format(input_seq[i], target_seq[i]))

Input Sequence: hey how are yo
Target Sequence: ey how are you
Input Sequence: good i am fine
Target Sequence: ood i am fine 
Input Sequence: have a nice da
Target Sequence: ave a nice day


In [7]:
for i in range(len(text)):
    input_seq[i] = [char2int[character] for character in input_seq[i]]
    target_seq[i] = [char2int[character] for character in target_seq[i]]

## One-Hot Encoding

Each character is represented as a one-hot vector of size `|V|` (vocabulary size).

Example (vocab size = 5):

'a' → [1, 0, 0, 0, 0]

'b' → [0, 1, 0, 0, 0]


### Limitations of One-Hot Encoding
- No semantic similarity between characters
- High dimensional and sparse
- Not scalable to large vocabularies

> Modern models replace this with **learned embeddings**.


Before encoding our input sequence into one-hot vectors, we'll define 3 key variables:

- *dict_size*: The number of unique characters that we have in our text
    - This will determine the one-hot vector size as each character will have an assigned index in that vector
- *seq_len*: The length of the sequences that we're feeding into the model
    - As we standardised the length of all our sentences to be equal to the longest sentences, this value will be the max length - 1 as we removed the last character input as well
- *batch_size*: The number of sentences that we defined and are going to feed into the model as a batch

In [8]:
dict_size = len(char2int)
seq_len = maxlen - 1
batch_size = len(text)

def one_hot_encode(sequence, dict_size, seq_len, batch_size):
    # Creating a multi-dimensional array of zeros with the desired output shape
    features = np.zeros((batch_size, seq_len, dict_size), dtype=np.float32)

    # Replacing the 0 at the relevant character index with a 1 to represent that character
    for i in range(batch_size):
        for u in range(seq_len):
            features[i, u, sequence[i][u]] = 1
    return features

We also defined a helper function that creates arrays of zeros for each character and replaces the corresponding character index with a **1**.

In [9]:
input_seq = one_hot_encode(input_seq, dict_size, seq_len, batch_size)
print("Input shape: {} --> (Batch Size, Sequence Length, One-Hot Encoding Size)".format(input_seq.shape))

Input shape: (3, 14, 17) --> (Batch Size, Sequence Length, One-Hot Encoding Size)


In [10]:
input_seq = torch.from_numpy(input_seq)
target_seq = torch.Tensor(target_seq)

In [11]:
is_cuda = torch.cuda.is_available()
if is_cuda:
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU not available, CPU used")

GPU not available, CPU used


In [12]:
class Model(nn.Module):
    def __init__(self, input_size, output_size, hidden_dim, n_layers):
        super(Model, self).__init__()

        # Defining some parameters
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        #Defining the layers
        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_dim, n_layers, batch_first=True)
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x):

        batch_size = x.size(0)

        #Initializing hidden state for first input using method defined below
        hidden = self.init_hidden(batch_size)

        # Passing in the input and hidden state into the model and obtaining outputs
        out, hidden = self.rnn(x, hidden)

        # Reshaping the outputs such that it can be fit into the fully connected layer
        out = out.contiguous().view(-1, self.hidden_dim)
        out = self.fc(out)

        return out, hidden

    def init_hidden(self, batch_size):
        # This method generates the first hidden state of zeros which we'll use in the forward pass
        hidden = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
         # We'll send the tensor holding the hidden state to the device we specified earlier as well
        return hidden

In [13]:
model = Model(input_size=dict_size, output_size=dict_size, hidden_dim=12, n_layers=1)
model = model.to(device)
n_epochs = 100
lr=0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

### Loss Function: Cross Entropy

We use `CrossEntropyLoss`, which combines:
- **Softmax** (to convert logits to probabilities)
- **Negative Log-Likelihood** (to measure prediction error)

For each time step, the loss is:

$$
\mathcal{L} = -\log P(y_{\text{true}})
$$

The loss is averaged across:
- All characters in the sequence
- All sequences in the batch


In [14]:
# Training Run
input_seq = input_seq.to(device)
for epoch in range(1, n_epochs + 1):
    optimizer.zero_grad() # Clears existing gradients from previous epoch
    #input_seq = input_seq.to(device)
    output, hidden = model(input_seq)
    output = output.to(device)
    target_seq = target_seq.to(device)
    loss = criterion(output, target_seq.view(-1).long())
    loss.backward() # Does backpropagation and calculates gradients
    optimizer.step() # Updates the weights accordingly

    if epoch%10 == 0:
        print('Epoch: {}/{}.............'.format(epoch, n_epochs), end=' ')
        print("Loss: {:.4f}".format(loss.item()))

Epoch: 10/100............. Loss: 2.4024
Epoch: 20/100............. Loss: 2.0977
Epoch: 30/100............. Loss: 1.6962
Epoch: 40/100............. Loss: 1.2881
Epoch: 50/100............. Loss: 0.9302
Epoch: 60/100............. Loss: 0.6478
Epoch: 70/100............. Loss: 0.4461
Epoch: 80/100............. Loss: 0.3109
Epoch: 90/100............. Loss: 0.2233
Epoch: 100/100............. Loss: 0.1681


## Training Observations

- Loss decreases smoothly → model is learning character transitions
- Model memorizes the training sentences
- Overfitting is expected due to the tiny dataset

This model does **not generalize**, but demonstrates how sequence modeling works.


## Text Generation (Inference)

During inference:
- The model uses its **own predictions** as the next input
- Errors may accumulate over time (exposure bias)

We use **greedy decoding** by selecting the most probable character.


In [16]:
def predict(model, character):
    # One-hot encoding our input to fit into the model
    character = np.array([[char2int[c] for c in character]])
    character = one_hot_encode(character, dict_size, character.shape[1], 1)
    character = torch.from_numpy(character)
    character = character.to(device)
    out, hidden = model(character)
    prob = nn.functional.softmax(out[-1], dim=0).data
    char_ind = torch.max(prob, dim=0)[1].item()
    return int2char[char_ind], hidden

In [17]:
def sample(model, out_len, start='hey'):
    model.eval() # eval mode
    start = start.lower()
    chars = [ch for ch in start]
    size = out_len - len(chars)
    for ii in range(size):
        char, h = predict(model, chars)
        chars.append(char)
    return ''.join(chars)

In [18]:
sample(model, 15, 'good')

'good i am fine '

## Limitations of Vanilla RNNs

- Vanishing and exploding gradients
- Difficulty learning long-term dependencies
- Sequential computation (slow)

These issues motivated the development of **LSTMs and GRUs**.


# Long Short-Term Memory (LSTM)

Vanilla RNNs struggle with **long-term dependencies** due to vanishing and exploding gradients. **Long Short-Term Memory (LSTM)** networks address this by explicitly **controlling information flow** across time steps.

The key innovation is the **cell state**, which acts as a long-term memory highway, allowing gradients to flow unchanged over many time steps.

---

## LSTM Components

At each time step `t`, an LSTM maintains:

- **Hidden state**: \( h_t \)
- **Cell state**: \( c_t \)


Information flow is regulated using **three gates**.

---

## Forget Gate

Determines what information from the previous cell state should be forgotten:

$$
f_t = \sigma(W_f [x_t, h_{t-1}] + b_f)
$$

- Output values in range \([0, 1]\)
- `0` → completely forget
- `1` → completely keep

---

## Input Gate

Controls what new information should be stored:

$$
i_t = \sigma(W_i [x_t, h_{t-1}] + b_i)
$$

Candidate memory:

$$
\tilde{c}_t = \tanh(W_c [x_t, h_{t-1}] + b_c)
$$

---

## Cell State Update

The cell state is updated as:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

- Old memory is selectively forgotten
- New memory is selectively added

This **additive update** helps prevent **vanishing gradients**.

---

## 4Output Gate

Controls what part of the cell state is exposed as hidden state:

$$
o_t = \sigma(W_o [x_t, h_{t-1}] + b_o)
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

---

## Why LSTMs Work Better Than RNNs

- **Explicit memory cell** for long-term dependencies
- **Gradient flow preserved** through additive updates
- **Better at capturing long-range dependencies**

---

## PyTorch Implementation

Replacing a vanilla RNN with an LSTM in PyTorch is straightforward:

```python
self.rnn = nn.LSTM(
    input_size,
    hidden_dim,
    num_layers,
    batch_first=True
)


# Gated Recurrent Unit (GRU)

Gated Recurrent Units (GRUs) were introduced as a **simplified alternative** to LSTMs, offering:
- Faster training
- Easier implementation
- Competitive performance

---

## GRU State Representation

GRUs maintain **only the hidden state**:
- $ h_t $

No separate cell state is used.

---

## Update Gate

Controls how much of the previous hidden state to retain:

$$
z_t = \sigma(W_z [x_t, h_{t-1}] + b_z)
$$

- Acts as a combination of LSTM's forget and input gates

---

## Reset Gate

Determines how much past information to ignore:

$$
r_t = \sigma(W_r [x_t, h_{t-1}] + b_r)
$$

---

## Candidate Hidden State

$$
\tilde{h}_t = \tanh(W_h [x_t, r_t \odot h_{t-1}] + b_h)
$$

---

## Final Hidden State Update

$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

---

## GRU vs LSTM

| Feature | LSTM | GRU |
|------|------|-----|
| Gates | 3 | 2 |
| Cell state | Yes | No |
| Parameters | More | Fewer |
| Training speed | Slower | Faster |

---

## PyTorch Implementation

```python
self.rnn = nn.GRU(
    input_size,
    hidden_dim,
    num_layers,
    batch_first=True
)

GRUs are often preferred when:

* Dataset is small
* Model size matters

# Transformer Architecture

Transformers were introduced to overcome the **sequential bottleneck** of RNN-based models.

**Key differences:**
- Process entire sequences **in parallel**
- Rely on **self-attention** instead of recurrence

---

## Core Idea: Self-Attention

Each token attends to **all other tokens** in the sequence, enabling direct modeling of long-range dependencies.

Given:
- Query $ Q $
- Key $ K $
- Value $ V $

Attention is computed as:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

---

## Why Self-Attention Works

- Learns **long-range dependencies directly**
- No vanishing gradient issues
- All-to-all token interaction

---

## Positional Encoding

Since Transformers have no recurrence, **positional information** is added explicitly:

$$
PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$
$$
PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

---

## Multi-Head Attention

Instead of a single attention operation, Transformers use **multiple heads**:

$$
\text{MultiHead}(Q,K,V) = \text{Concat}(head_1,\dots,head_h)W^O
$$

Each head learns a different relationship:
- Syntax
- Semantics
- Positional relations

---

## Transformer Block

Each block contains:
1. **Multi-head self-attention**
2. **Add & LayerNorm**
3. **Feedforward network**
4. **Add & LayerNorm**

---

## Encoder vs Decoder

- Encoder: BERT (bidirectional)
- Decoder: GPT (causal, autoregressive)
- Encoder–Decoder: T5, original Transformer

---

## Why Transformers Replaced RNNs

| Aspect | RNN/LSTM | Transformer |
|-----|--------|------------|
| Parallelism | No | Yes |
| Long-range deps | Difficult | Easy |
| Training speed | Slow | Fast |
| Scaling | Limited | Excellent |

---

## Connection to This Notebook

This character-level RNN learns the probability distribution:

$$
P(x_t | x_{<t})
$$

**GPT-style Transformers** learn the **same objective**, but with key differences:

- **Attention** instead of recurrence
- **Massive scale** (more parameters, data, and compute)
- **Subword tokenization** (e.g., Byte Pair Encoding)


## To-Do Questions

1. **(Conceptual)**  
   Vanilla RNNs struggle with long-term dependencies due to the vanishing gradient problem.  
   Understand this problem mathematically and describe how the LSTM architecture (using gates and the cell state) helps mitigate it.

2. **(Coding)**  
   Replace the `nn.RNN` layer in this notebook with an `nn.LSTM` **without changing any other part of the code**.  
   Train the model and compare:
   - Training loss convergence
   - Quality of generated text  
   Briefly explain any differences you observe.


# Task
Modify the `Model` class to use `nn.LSTM` instead of `nn.RNN`, then reinitialize and train the model, observe loss convergence, generate text using the trained LSTM model, and finally, compare the training loss convergence and generated text quality between the LSTM and the previous RNN model.

In [19]:
class Model(nn.Module):
    def __init__(self, input_size, output_size, hidden_dim, n_layers):
        super(Model, self).__init__()

        # Defining some parameters
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        #Defining the layers
        # RNN Layer (now LSTM Layer)
        self.rnn = nn.LSTM(input_size, hidden_dim, n_layers, batch_first=True)
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x):

        batch_size = x.size(0)

        #Initializing hidden state for first input using method defined below
        # LSTM returns (h_n, c_n) so init_hidden needs to return a tuple too
        hidden = self.init_hidden(batch_size)

        # Passing in the input and hidden state into the model and obtaining outputs
        out, hidden = self.rnn(x, hidden)

        # Reshaping the outputs such that it can be fit into the fully connected layer
        out = out.contiguous().view(-1, self.hidden_dim)
        out = self.fc(out)

        return out, hidden

    def init_hidden(self, batch_size):
        # This method generates the first hidden state (h0) and cell state (c0) of zeros
        # which we'll use in the forward pass for LSTM
        h0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        return (h0, c0)

In [20]:
model = Model(input_size=dict_size, output_size=dict_size, hidden_dim=12, n_layers=1)
model = model.to(device)
n_epochs = 100
lr=0.01
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training Run
input_seq = input_seq.to(device)

# Lists to store training progress for comparison
lstm_losses = []

print("\n--- Training LSTM Model ---")
for epoch in range(1, n_epochs + 1):
    optimizer.zero_grad() # Clears existing gradients from previous epoch
    output, hidden = model(input_seq)
    output = output.to(device)
    target_seq = target_seq.to(device)
    loss = criterion(output, target_seq.view(-1).long())
    loss.backward() # Does backpropagation and calculates gradients
    optimizer.step() # Updates the weights accordingly

    if epoch%10 == 0:
        print('Epoch: {}/{}.............'.format(epoch, n_epochs), end=' ')
        print("Loss: {:.4f}".format(loss.item()))
        lstm_losses.append(loss.item())


--- Training LSTM Model ---
Epoch: 10/100............. Loss: 2.6448
Epoch: 20/100............. Loss: 2.3912
Epoch: 30/100............. Loss: 2.1474
Epoch: 40/100............. Loss: 1.8178
Epoch: 50/100............. Loss: 1.4731
Epoch: 60/100............. Loss: 1.1477
Epoch: 70/100............. Loss: 0.8632
Epoch: 80/100............. Loss: 0.6422
Epoch: 90/100............. Loss: 0.4748
Epoch: 100/100............. Loss: 0.3517


In [21]:
print("Generated text with LSTM:")
print(sample(model, 15, 'good'))

Generated text with LSTM:
good i am fine 


## Comparison: RNN vs. LSTM

### Training Loss Convergence
- **RNN Loss (last epoch):** 0.1681
- **LSTM Loss (last epoch):** 0.3517

**Observation:** In this specific experiment with a very small dataset and limited epochs, the vanilla RNN achieved a slightly lower final training loss than the LSTM. This is likely due to the RNN's simpler architecture being sufficient to memorize the tiny training set, while the LSTM's additional gates might introduce a bit more complexity or require more data/epochs to fully show its benefits on this specific task.

### Quality of Generated Text
- **RNN Generated Text (from `sample(model, 15, 'good')`):** 'good i am fine '
- **LSTM Generated Text (from `sample(model, 15, 'good')`):** 'good i am fine '

**Observation:** Both the RNN and LSTM models generated identical text, which perfectly reproduced one of the training sentences ('good i am fine'). This indicates that both models successfully *memorized* the training data. Given the very small and repetitive nature of the training data, both architectures were able to fit the patterns well for this specific inference prompt. For more complex, longer sequences and larger vocabularies, LSTMs would typically outperform vanilla RNNs in generating more coherent and contextually accurate text due to their ability to handle long-term dependencies.

### Conclusion
For this extremely limited dataset, both RNN and LSTM models performed similarly in memorizing the short training phrases and generating them accurately. The slight difference in final loss is minor and context-dependent. The benefits of LSTMs over RNNs (better handling of vanishing gradients and long-term dependencies) would become more apparent with larger, more complex sequence data.